In [9]:
!pip install torch tokenizers tqdm

In [10]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

In [11]:
class TextDataset(Dataset):
    def __init__(self, text_file, seq_length=128):
        with open("alice.txt", 'r', encoding="utf-8") as f:
            text = f.read()
        words = text.split()

        unique_words = sorted(set(words))
        self.word_to_idx = {word: i for i, word in enumerate(unique_words)}
        self.idx_to_word = {i: word for i, word in enumerate(unique_words)}
        self.vocab_size = len(unique_words)

        self.data = [self.word_to_idx[word] for word in words]
        self.seq_length = seq_length
    def __len__(self):
        return len(self.data) - self.seq_length
    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx:idx+self.seq_length])
        y = torch.tensor(self.data[idx+1:idx+self.seq_length+1])
        return x, y

In [12]:
class TransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4, dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = nn.Embedding(5000, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.fc = nn.Linear(d_model, vocab_size)
        self.d_model = d_model
    def forward(self, x):
        seq_len = x.size(1)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        x = self.embedding(x) + self.pos_encoder(positions)
        x = x.transpose(0, 1) 
        x = self.transformer(x)
        x = x.transpose(0, 1)
        return self.fc(x) # Transformer expects (seq_len, batch_size, d_model)

In [ ]:
from tqdm import tqdm

dataset = TextDataset('alice.txt')
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

model = TransformerLM(vocab_size=dataset.vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()  

for epoch in range(10):
    progress_bar = tqdm(dataloader, desc=f'Epoch {epoch}')
    # for epoch in range(10):
    for x, y in dataloader:
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output.view(-1, dataset.vocab_size), y.view(-1))
        loss.backward()
        optimizer.step()

        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
    print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

c:\Users\mason\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
Epoch 0:   0%|          | 0/920 [06:12<?, ?it/s, loss=0.1463]


KeyboardInterrupt: 

# Adjustments to run quicker:

In [13]:
import torch
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
import os

In [28]:
class TextDataset(Dataset):
    def __init__(self, text_file, seq_length=128):
        with open(text_file, 'r', encoding='utf-8') as f:
            text = f.read()

        # Byte Pair Encoding Tokenizer forr Subword Tokenization
        if not os.path.exists("tokenizer.json"):
            tokenizer = Tokenizer(models.BPE())
            tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

            trainer = trainers.BpeTrainer(special_tokens=["<PAD>", "<UNK>"], vocab_size=1000)
            tokenizer.train_from_iterator([text], trainer)
            tokenizer.save("tokenizer.json")

        else:
            tokenizer = Tokenizer.from_file("tokenizer.json")

        # Encode text
        encoded = tokenizer.encode(text)
        self.tokenizer = tokenizer
        self.input_ids = encoded.ids
        self.vocab_size = tokenizer.get_vocab_size()

        self.seq_length = seq_length

    def __len__(self):
        return len(self.input_ids) - self.seq_length

    def __getitem__(self, idx):
        x = torch.tensor(self.input_ids[idx:idx+self.seq_length])
        y = torch.tensor(self.input_ids[idx+1:idx+self.seq_length+1])
        return x, y

In [33]:
import torch.nn as nn

class TransformerLM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, n_heads=4, num_layers=2, max_seq_len=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.positional_encoding = nn.Parameter(torch.zeros(1, max_seq_len, embed_dim))

        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=n_heads)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)
        x = self.embedding(x) + self.positional_encoding[:, :seq_len, :]
        x = x.transpose(0, 1)  # Transformer expects [seq_len, batch, dim]
        out = self.transformer(x)
        out = out.transpose(0, 1)  # Back to [batch, seq_len, dim]
        logits = self.fc_out(out)
        return logits


In [34]:
from tqdm import tqdm

def train_model(text_file='alice.txt', epochs=1, seq_length=64, batch_size=2, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    dataset = TextDataset(text_file, seq_length)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = TransformerLM(vocab_size=dataset.vocab_size).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    print(f"Vocab size: {dataset.vocab_size}")

    for epoch in range(epochs):
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
        total_loss = 0.0

        for x, y in progress_bar:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()

            output = model(x)
            loss = criterion(output.view(-1, dataset.vocab_size), y.view(-1))
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1} complete — avg loss: {avg_loss:.4f}\n")

    return model, dataset

In [39]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: False
GPU name: None


In [36]:
if __name__ == "__main__":
    model, dataset = train_model('alice.txt', epochs=1, seq_length=64, batch_size=8, lr=1e-3)


Using device: cpu
Vocab size: 4000


Epoch 1/1: 100%|██████████| 4953/4953 [07:05<00:00, 11.65it/s, loss=0.9950]


Epoch 1 complete — avg loss: 1.8348

